# Day 12 — Pretrained Encoders and Multispectral Channel Adaptation

Day 5 established that ImageNet pretraining is worth more than architecture: a frozen
ResNet-18 backbone beat a from-scratch CNN while training 0.046% of the network, and
fine-tuning reached 93.8% against a 76.6% baseline.

It also flagged the obstacle that blocks the same move on satellite data:

> A 13-band Sentinel-2 stack does not fit a 3-channel first conv layer.

Day 10 hit it again from the other side — `UNet(in_ch=4)` costs one argument when
training from scratch, and is impossible with pretrained weights.

This notebook measures the three ways around it.

## The obstacle, concretely

ResNet-18's first layer has weight shape `[64, 3, 7, 7]`: 64 filters, each spanning
**exactly three** input channels. Four bands will not go in. Neither will thirteen.

## Three strategies

| | what it does | what it costs |
|---|---|---|
| **RGB only** | drop the extra bands | throws away the bands that usually discriminate best — NIR and SWIR for ice, snow, water and vegetation |
| **Re-initialize** | replace `conv1` with a fresh random N-channel layer | keeps every pretrained layer but the first; that first layer is the generic edge-and-colour bank Day 5 showed is the most transferable part |
| **Inflate** | copy the pretrained RGB kernels and extend them across the extra channels | keeps the spatial prior; the extra channels start as an average of the RGB filters, which is an assumption, not a fact |

Plus a from-scratch arm for reference.

## Result

Three seeds per arm, evaluated on a geographically held-out region:

| strategy | TEST mIoU (mean) | std | spread |
|---|---|---|---|
| scratch (4 bands, no pretraining) | **0.5092** | 0.0074 | 0.5006–0.5144 |
| reinit (4 bands, pretrained except conv1) | 0.4993 | 0.0439 | 0.4499–0.5338 |
| inflate (4 bands, conv1 inflated) | 0.4915 | 0.0396 | 0.4480–0.5253 |
| **rgb_only (3 bands, fully pretrained)** | **0.2396** | 0.0190 | 0.2229–0.2603 |

**One result is established and the rest are not.** Discarding NIR to fit a 3-channel
pretrained layer costs **27 points** — six to ten standard deviations. The other three
arms are mutually indistinguishable, and the from-scratch arm has the highest mean.

The strategy this notebook was built to evaluate, `inflate` versus `reinit`, differs by
−0.008 ± 0.023 and **changes sign across seeds**. The pretrained first layer has no
measurable value here.

An unanticipated observation: the pretrained arms are **five times more variable** than
the from-scratch arm (std 0.040–0.044 against 0.007). Same architecture, same parameter
count — the only difference is initialization.

Section 7 argues most of this negative result is a property of synthetic data and says
which parts transfer.

## Method note

Evaluation uses the **blocked split from Day 11**, not a random one. Day 11 measured
that random tile splitting inflates validation mIoU by 16.5 points on this scene. There
is no reason to adopt a comparison protocol already shown to be wrong.

## 0. Setup

```bash
conda activate dl
python -m pip install segmentation-models-pytorch
```

`segmentation_models_pytorch` provides U-Nets with interchangeable pretrained encoders.
It will also do channel adaptation silently if given `in_channels=4` — this notebook
does it explicitly instead, because the three strategies give different results and the
default should be a choice rather than a surprise.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import time, copy
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.windows import Window
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print("device:", device, "| smp", smp.__version__)

WORK = os.path.expanduser("~/dl-local/geo")
os.makedirs(WORK, exist_ok=True)
IGNORE, N_CLS, TILE = 255, 3, 128
CLS_NAMES = ["low", "mid", "high"]

def seed_all(s=42):
    torch.manual_seed(s)
    if device == "mps":   torch.mps.manual_seed(s)
    elif device == "cuda": torch.cuda.manual_seed_all(s)

In [ ]:
import torchvision
w = torchvision.models.resnet18(weights="IMAGENET1K_V1").conv1.weight
print("ResNet-18 conv1 weight shape:", tuple(w.shape))
print("  64 filters, 7x7 spatial, exactly 3 input channels")
print("\nA 4-band stack cannot be fed to this layer. Neither can Sentinel-2's 13.")

## 1. A scene where the extra band matters

Day 11's scene will not work for this question: all four of its bands were linear
rescalings of one underlying field, so dropping NIR would lose nothing and the
"RGB only" arm would score as well as the rest. A comparison that cannot distinguish
the strategies would tell us nothing — the same trap as Day 11 §1b.

This scene gives the visible bands and the NIR band **different** underlying fields, and
makes the label depend on both. Now discarding NIR is a real loss, which is the
situation that motivates channel adaptation in the first place.

The `hidden` component from Day 11 is kept, so the task stays non-trivial.

In [ ]:
def make_multispectral_scene(path, lbl_path, H=2048, W=2048, seed=0):
    """Visible bands and NIR carry DIFFERENT fields; labels depend on both.

    vis   -> bands 0,1,2   (what an RGB-only model sees)
    nir   -> band 3        (unique information, lost if dropped)
    hidden-> neither       (keeps the task non-pointwise-invertible, as in Day 11)
    """
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]

    def field(sigma, off):
        r = np.random.default_rng(seed + off).normal(0, 1, (H, W))
        f = gaussian_filter(r, sigma=sigma)
        return (f - f.mean()) / (f.std() + 1e-9)

    vis    = 0.6 * field(40, 1) + 0.9 * field(240, 2)
    nir    = 0.6 * field(40, 3) + 0.9 * field(240, 4)      # independent of vis
    hidden = field(70, 5)                                   # in neither band set
    trend  = (yy / H - 0.5) * 1.6 + (xx / W - 0.5) * 0.6

    label_signal = 0.75 * vis + 0.75 * nir + 0.7 * hidden + trend

    bands = np.stack([
        vis * 1100 + 3000 + rng.normal(0, 90,  (H, W)),
        vis *  850 + 2200 + rng.normal(0, 75,  (H, W)),
        vis *  550 + 1500 + rng.normal(0, 60,  (H, W)),
        nir * 2300 + 8000 + rng.normal(0, 220, (H, W)),     # the informative extra band
    ]).astype(np.int16)

    nod = (yy + xx) < 400
    bands[:, nod] = -9999
    valid = ~nod

    q = np.quantile(label_signal[valid], [0.33, 0.70])
    lab = np.full((H, W), IGNORE, dtype=np.uint8)
    lab[valid & (label_signal <  q[0])] = 0
    lab[valid & (label_signal >= q[0]) & (label_signal < q[1])] = 1
    lab[valid & (label_signal >= q[1])] = 2

    tr = from_origin(499980.0, 5600040.0, 10.0, 10.0)
    prof = dict(driver="GTiff", height=H, width=W, crs="EPSG:32633", transform=tr,
                tiled=True, blockxsize=256, blockysize=256, compress="deflate")
    with rasterio.open(path, "w", count=4, dtype="int16", nodata=-9999, **prof) as d:
        d.write(bands)
    with rasterio.open(lbl_path, "w", count=1, dtype="uint8", nodata=IGNORE, **prof) as d:
        d.write(lab, 1)

    v = valid.ravel()
    ls = label_signal.ravel()[v]
    r2_vis = np.corrcoef(vis.ravel()[v], ls)[0, 1] ** 2
    r2_nir = np.corrcoef(nir.ravel()[v], ls)[0, 1] ** 2
    r2_both = np.corrcoef((0.75*vis + 0.75*nir).ravel()[v], ls)[0, 1] ** 2
    return path, lbl_path, dict(vis=r2_vis, nir=r2_nir, both=r2_both)


scene, labels, r2 = make_multispectral_scene(
    f"{WORK}/ms_scene.tif", f"{WORK}/ms_labels.tif")
print("wrote", scene)
print("\nlabel variance explained by:")
print(f"  visible bands alone : R^2 = {r2['vis']:.3f}")
print(f"  NIR alone           : R^2 = {r2['nir']:.3f}")
print(f"  visible + NIR       : R^2 = {r2['both']:.3f}")
print("\nthe gap between the first and third rows is what an RGB-only model cannot reach")

The gap between "visible alone" and "visible + NIR" is the ceiling an RGB-only model
is capped at, before any question of architecture or pretraining.

This is the quantitative version of a routine remote-sensing fact: the discriminative
bands for ice, snow, water and vegetation are usually **not** in the visible range.
Dropping them to fit a pretrained layer trades a real information loss for convenience.

## 2. Blocked split (Day 11's protocol)

Southern quarter held out as TEST, northern pool split into spatial blocks with a buffer
derived from the measured autocorrelation range.

In [ ]:
def semivariogram_range(arr, valid, max_lag=400, n_lags=25, n_pairs=40000, seed=0):
    rng = np.random.default_rng(seed); H, W = arr.shape
    edges = np.linspace(1, max_lag, n_lags + 1)
    sums = np.zeros(n_lags); counts = np.zeros(n_lags)
    for _ in range(n_pairs):
        r1 = rng.integers(0, H); c1 = rng.integers(0, W)
        if not valid[r1, c1]: continue
        h = rng.uniform(1, max_lag); th = rng.uniform(0, 2*np.pi)
        r2 = int(r1 + h*np.sin(th)); c2 = int(c1 + h*np.cos(th))
        if not (0 <= r2 < H and 0 <= c2 < W) or not valid[r2, c2]: continue
        d = np.hypot(r2-r1, c2-c1); b = np.searchsorted(edges, d) - 1
        if 0 <= b < n_lags:
            sums[b] += (float(arr[r1,c1]) - float(arr[r2,c2]))**2; counts[b] += 1
    centers = 0.5*(edges[:-1] + edges[1:])
    g = np.where(counts > 0, 0.5*sums/np.maximum(counts,1), np.nan)
    sill = np.nanmean(g[-5:]); idx = np.where(g >= 0.95*sill)[0]
    return float(centers[idx[0]]) if len(idx) else float("nan")


def enumerate_tiles(p, size=TILE, stride=TILE, min_valid=0.6):
    with rasterio.open(p) as src:
        H, W, nodv = src.height, src.width, src.nodata
        out = []
        for r in range(0, H-size+1, stride):
            for c in range(0, W-size+1, stride):
                if (src.read(1, window=Window(c,r,size,size)) != nodv).mean() >= min_valid:
                    out.append((c, r))
    return out, H, W


def split_blocked(tiles, block_tiles=4, val_frac=0.2, buffer_px=0, seed=0):
    rng = np.random.default_rng(seed); bsz = block_tiles * TILE
    blocks = {}
    for c, r in tiles:
        blocks.setdefault((c//bsz, r//bsz), []).append((c, r))
    keys = sorted(blocks)
    k = max(1, int(round(val_frac * len(keys))))
    val_keys = set(map(tuple, np.array(keys)[rng.permutation(len(keys))[:k]].tolist()))
    val = [t for kk in val_keys for t in blocks[kk]]
    tr_all = [t for kk in keys if kk not in val_keys for t in blocks[kk]]
    if buffer_px > 0 and val:
        vc = np.array([[c+TILE/2, r+TILE/2] for c, r in val])
        tr = [(c,r) for c,r in tr_all
              if np.hypot(vc[:,0]-(c+TILE/2), vc[:,1]-(r+TILE/2)).min() > buffer_px]
    else:
        tr = tr_all
    return tr, val


with rasterio.open(scene) as src:
    b0 = src.read(1).astype(np.float32); nodv = src.nodata
RANGE_PX = semivariogram_range(b0, b0 != nodv)
BUFFER = float(np.ceil(RANGE_PX / TILE) * TILE)

all_tiles, H, W = enumerate_tiles(scene)
TEST_ROW0 = 1536
pool = [(c,r) for c,r in all_tiles if r + TILE <= TEST_ROW0]
test_tiles = [(c,r) for c,r in all_tiles if r >= TEST_ROW0]
train_tiles, val_tiles = split_blocked(pool, 4, 0.2, BUFFER, seed=0)

print(f"range {RANGE_PX:.0f} px -> buffer {BUFFER:.0f} px")
print(f"train {len(train_tiles)}  val {len(val_tiles)}  TEST {len(test_tiles)}")

## 3. Dataset, and a deliberate break with Day 5

Day 5's rule was to respect the pretrained preprocessing contract: ImageNet weights
expect ImageNet channel statistics.

**That rule is applied differently here, on purpose.** ImageNet's statistics describe
photographic sRGB. These bands are int16 reflectance whose "red" is not a camera's red,
and there is no ImageNet statistic at all for the fourth channel. Forcing ImageNet
constants onto them would be numerology.

What the contract actually requires is that inputs arrive in **roughly the standardized
range the weights were trained on** — centred near zero, unit-ish scale — not that they
carry ImageNet's exact numbers. Per-band standardization from the training tiles
satisfies that and is what practitioners do on satellite data.

The thing that would genuinely break the contract is feeding raw reflectance in the
thousands. That is the error Day 5 was warning about.

In [ ]:
def band_stats(path, tiles, size=TILE, max_tiles=60, seed=0):
    """Per-band mean/std from TRAINING tiles only (Day 11: scene-wide stats leak)."""
    rng = np.random.default_rng(seed)
    pick = [tiles[i] for i in rng.permutation(len(tiles))[:max_tiles]]
    with rasterio.open(path) as src:
        nodv = src.nodata
        acc = [[] for _ in range(src.count)]
        for c, r in pick:
            a = src.read(window=Window(c, r, size, size)).astype(np.float64)
            for b in range(src.count):
                v = a[b][a[b] != nodv]
                if v.size: acc[b].append(v)
    return {b: dict(mean=float(np.concatenate(ch).mean()),
                    std=float(np.concatenate(ch).std()))
            for b, ch in enumerate(acc)}


class MSTileDS(Dataset):
    def __init__(self, scene_path, label_path, tiles, stats, bands=(0,1,2,3), size=TILE):
        self.scene, self.label_path = scene_path, label_path
        self.tiles, self.stats, self.bands, self.size = tiles, stats, bands, size
        with rasterio.open(scene_path) as src:
            self.nod = src.nodata

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        c, r = self.tiles[i]
        w = Window(c, r, self.size, self.size)
        with rasterio.open(self.scene) as src:
            arr = src.read(window=w).astype(np.float32)
        with rasterio.open(self.label_path) as ls:
            y = ls.read(1, window=w).astype(np.int64)
        valid = (arr != self.nod).all(axis=0)
        out = np.stack([(arr[b] - self.stats[b]["mean"]) / (self.stats[b]["std"] + 1e-8)
                        for b in self.bands])
        x = torch.from_numpy(out)
        x[:, ~valid] = 0.0
        y[~valid] = IGNORE
        return x, torch.from_numpy(y)


STATS = band_stats(scene, train_tiles)
for b, s in STATS.items():
    print(f"  band {b}: mean {s['mean']:8.1f}  std {s['std']:7.1f}")

## 4. The three adaptation strategies

**Inflate** is the one worth reading closely:

```python
mean_k = w.mean(dim=1, keepdim=True)              # [64,1,7,7] -- average RGB kernel
new_w  = torch.cat([w, mean_k.repeat(1, n_extra, 1, 1)], dim=1)
new_w *= 3.0 / in_ch                               # preserve activation magnitude
```

Each extra channel starts as the **mean of the three pretrained RGB kernels**, which
preserves the learned spatial structure (edge and blob detectors) while making no claim
about that band's colour response.

The `3/in_ch` rescaling matters because convolution sums over input channels: four
channels through unscaled kernels produce activations about 4/3 too large, and the
pretrained BatchNorm running statistics immediately after `conv1` were calibrated for
the original magnitude. **With a frozen backbone or a very low learning rate, skipping
this scaling measurably hurts.** With full fine-tuning, BatchNorm adapts within an epoch
or two and the effect is small — but it costs one line.

In [ ]:
def make_model(strategy, in_ch=4, n_classes=N_CLS, encoder="resnet18"):
    """Build a U-Net with a ResNet encoder under one of four channel strategies."""
    if strategy == "scratch":
        return smp.Unet(encoder, encoder_weights=None, in_channels=in_ch,
                        classes=n_classes), (0, 1, 2, 3)

    if strategy == "rgb_only":
        return smp.Unet(encoder, encoder_weights="imagenet", in_channels=3,
                        classes=n_classes), (0, 1, 2)

    m = smp.Unet(encoder, encoder_weights="imagenet", in_channels=3, classes=n_classes)
    old = m.encoder.conv1
    new = nn.Conv2d(in_ch, old.out_channels, kernel_size=old.kernel_size,
                    stride=old.stride, padding=old.padding, bias=old.bias is not None)

    if strategy == "inflate":
        w = old.weight.data                                   # [64, 3, 7, 7]
        mean_k = w.mean(dim=1, keepdim=True)                  # [64, 1, 7, 7]
        new_w = torch.cat([w, mean_k.repeat(1, in_ch - 3, 1, 1)], dim=1)
        new.weight.data = new_w * (3.0 / in_ch)               # preserve magnitude
    # strategy == "reinit": leave `new` at its random initialization

    m.encoder.conv1 = new
    return m, (0, 1, 2, 3)


# Inspect what each strategy produces in conv1
for s in ["scratch", "rgb_only", "reinit", "inflate"]:
    m, bands = make_model(s)
    w = m.encoder.conv1.weight.data
    print(f"{s:9s} conv1 {str(tuple(w.shape)):16s} bands {bands}  "
          f"std {w.std():.4f}  mean|w| {w.abs().mean():.4f}")

The `std` column is the diagnostic. A randomly initialized `conv1` and a pretrained one
have visibly different weight statistics — pretrained filters are structured, random
ones are not. `inflate` should sit near the pretrained values scaled by 3/4; `reinit`
should look like `scratch`'s first layer.

## 5. Train all four

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE)

@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    cm = torch.zeros(N_CLS, N_CLS, dtype=torch.int64, device=device)
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1)
        keep = yb != IGNORE
        y, p = yb[keep], pred[keep]
        cm += torch.bincount(y * N_CLS + p, minlength=N_CLS**2).reshape(N_CLS, N_CLS)
    cm = cm.float(); tp = cm.diag()
    iou = tp / (cm.sum(0) + cm.sum(1) - tp).clamp(min=1)
    present = cm.sum(1) > 0
    return dict(acc=(tp.sum()/cm.sum()).item(),
                miou=iou[present].mean().item(), iou=iou.cpu().numpy())


def run(strategy, epochs=15, lr=3e-4, seed=42):
    seed_all(seed)
    model, bands = make_model(strategy)
    model = model.to(device)

    tr_dl = DataLoader(MSTileDS(scene, labels, train_tiles, STATS, bands),
                       batch_size=8, shuffle=True, num_workers=0)
    va_dl = DataLoader(MSTileDS(scene, labels, val_tiles, STATS, bands),
                       batch_size=8, shuffle=False, num_workers=0)
    te_dl = DataLoader(MSTileDS(scene, labels, test_tiles, STATS, bands),
                       batch_size=8, shuffle=False, num_workers=0)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best = {"miou": -1}
    for ep in range(epochs):
        t0 = time.time(); model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        v = evaluate(model, va_dl)
        if v["miou"] > best["miou"]:
            best, best_ep = v, ep + 1
            best_state = copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()})
        if ep == 0 or (ep + 1) % 5 == 0:
            print(f"  [{strategy}] ep {ep+1:2d}  val mIoU {v['miou']:.4f}  ({time.time()-t0:.0f}s)")
    model.load_state_dict(best_state)
    te = evaluate(model, te_dl)
    print(f"  [{strategy}] best val {best['miou']:.4f} @ ep {best_ep}  ->  "
          f"TEST {te['miou']:.4f}")
    return dict(name=strategy, val=best["miou"], test=te["miou"], test_iou=te["iou"])


results = []
for s in ["scratch", "rgb_only", "reinit", "inflate"]:
    print(s)
    results.append(run(s))
    print()

scratch
  [scratch] ep  1  val mIoU 0.3927  (5s)
  [scratch] ep  5  val mIoU 0.4699  (1s)
  [scratch] ep 10  val mIoU 0.4969  (1s)
  [scratch] ep 15  val mIoU 0.4999  (1s)
  [scratch] best val 0.5250 @ ep 12  ->  TEST 0.5006

rgb_only
  [rgb_only] ep  1  val mIoU 0.2294  (1s)
  [rgb_only] ep  5  val mIoU 0.2905  (1s)
  [rgb_only] ep 10  val mIoU 0.2815  (1s)
  [rgb_only] ep 15  val mIoU 0.2965  (1s)
  [rgb_only] best val 0.3503 @ ep 9  ->  TEST 0.2229

reinit
  [reinit] ep  1  val mIoU 0.2946  (1s)
  [reinit] ep  5  val mIoU 0.4154  (1s)
  [reinit] ep 10  val mIoU 0.4463  (1s)
  [reinit] ep 15  val mIoU 0.4758  (1s)
  [reinit] best val 0.4811 @ ep 13  ->  TEST 0.5140

inflate
  [inflate] ep  1  val mIoU 0.2469  (1s)
  [inflate] ep  5  val mIoU 0.4451  (1s)
  [inflate] ep 10  val mIoU 0.4437  (1s)
  [inflate] ep 15  val mIoU 0.5168  (1s)
  [inflate] best val 0.5168 @ ep 15  ->  TEST 0.5253


In [ ]:
import pandas as pd

rows = []
for r in results:
    row = {"strategy": r["name"], "val mIoU": round(r["val"], 4),
           "TEST mIoU": round(r["test"], 4)}
    for i, c in enumerate(CLS_NAMES):
        row[f"TEST IoU {c}"] = round(float(r["test_iou"][i]), 4)
    rows.append(row)
df = pd.DataFrame(rows).set_index("strategy")
print(df)

base = df.loc["scratch", "TEST mIoU"]
print("\nTEST mIoU relative to from-scratch:")
for s in df.index:
    print(f"  {s:9s} {df.loc[s,'TEST mIoU']:.4f}  ({df.loc[s,'TEST mIoU']-base:+.4f})")

## 6. A single seed cannot answer the question this notebook asks

From the run above:

```
inflate  TEST 0.5253
reinit   TEST 0.5140     difference 1.1 points
```

Day 6 measured run-to-run variation of ~1.2 points on a comparable setup. **The
difference this notebook exists to measure is smaller than the noise floor.** Reporting
`inflate` as the winner on that basis would be reading a coin flip.

Unlike Day 9 — where resolving an ambiguous ranking would have cost 80 minutes and the
conclusions did not depend on it — each arm here takes about 20 seconds. Three seeds
across four arms is roughly four minutes, and it answers the central question. That is
worth doing.

In [ ]:
# The inflate-vs-reinit gap sits inside Day 6's measured run-to-run variance,
# so a single seed cannot resolve it. Four arms at ~20 s each makes three seeds
# cheap -- unlike Day 9, where the same question would have cost 80 minutes.
multi = {}
for s in ["scratch", "rgb_only", "reinit", "inflate"]:
    multi[s] = []
    for sd in [42, 0, 7]:
        seed_all(sd)
        model, bands = make_model(s)
        model = model.to(device)
        tr = DataLoader(MSTileDS(scene, labels, train_tiles, STATS, bands),
                        batch_size=8, shuffle=True, num_workers=0)
        va = DataLoader(MSTileDS(scene, labels, val_tiles, STATS, bands),
                        batch_size=8, shuffle=False, num_workers=0)
        te = DataLoader(MSTileDS(scene, labels, test_tiles, STATS, bands),
                        batch_size=8, shuffle=False, num_workers=0)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)
        best = {"miou": -1}
        for ep in range(15):
            model.train()
            for xb, yb in tr:
                xb, yb = xb.to(device), yb.to(device)
                loss = loss_fn(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()
            v = evaluate(model, va)
            if v["miou"] > best["miou"]:
                best = v
                bs = copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()})
        model.load_state_dict(bs)
        multi[s].append(evaluate(model, te)["miou"])
        print(f"  {s:9s} seed {sd:2d}  TEST {multi[s][-1]:.4f}")
    print()

print(f"{'strategy':10s} {'mean':>8s} {'std':>8s} {'range':>18s}")
for s, v in multi.items():
    a = np.array(v)
    print(f"{s:10s} {a.mean():8.4f} {a.std(ddof=1):8.4f} "
          f"{a.min():8.4f}-{a.max():.4f}")

d = np.array(multi["inflate"]) - np.array(multi["reinit"])
print(f"\ninflate - reinit per seed: {np.round(d, 4).tolist()}")
print(f"  mean {d.mean():+.4f}  std {d.std(ddof=1):.4f}")

  scratch   seed 42  TEST 0.5006
  scratch   seed  0  TEST 0.5144
  scratch   seed  7  TEST 0.5125

  rgb_only  seed 42  TEST 0.2229
  rgb_only  seed  0  TEST 0.2356
  rgb_only  seed  7  TEST 0.2603

  reinit    seed 42  TEST 0.5140
  reinit    seed  0  TEST 0.5338
  reinit    seed  7  TEST 0.4499

  inflate   seed 42  TEST 0.5253
  inflate   seed  0  TEST 0.5012
  inflate   seed  7  TEST 0.4480

strategy       mean      std              range
scratch      0.5092   0.0074   0.5006-0.5144
rgb_only     0.2396   0.0190   0.2229-0.2603
reinit       0.4993   0.0439   0.4499-0.5338
inflate      0.4915   0.0396   0.4480-0.5253

inflate - reinit per seed: [0.0113, -0.0326, -0.002]
  mean -0.0078  std 0.0225


## 7. Reading the result

### What is established

**Discarding NIR costs 27 points.** `rgb_only` reaches 0.2396 against ~0.49–0.51 for the
4-band arms — six to ten standard deviations, on any arm's noise estimate. Section 1
printed the mechanism: the visible bands explain R² ≈ 0.26 of the label, all four
explain ≈ 0.74. The measurement matches the information content.

For satellite work the implication is blunt. **When the choice is between using a
pretrained backbone and keeping every band, keep the bands.** The discriminative
channels for ice, snow, water and vegetation are mostly outside the visible range, and
dropping them to fit a 3-channel layer is a far worse trade than losing pretrained
weights.

### What is not established

**`inflate` versus `reinit`: −0.008 ± 0.023, sign flips across seeds.** This is the
comparison the notebook was built for, and the answer is that the pretrained first layer
has no measurable value on this data. The kernel-averaging and the `3/in_ch` rescaling —
the most carefully reasoned code here — made no difference that could be detected.

**Any pretrained arm versus from-scratch.** `scratch` has the *highest* mean (0.5092)
and `inflate` the lowest of the three 4-band arms (0.4915), separated by 1.8 points
against a pooled standard deviation near 2.9. All three are tied. **ImageNet pretraining
produced no measurable benefit here at all.**

### An unanticipated observation

```
scratch   std 0.0074
reinit    std 0.0439    5.9x
inflate   std 0.0396    5.4x
```

All four arms share an architecture and a parameter count; the only difference is how
the weights start. **Initializing from ImageNet on data that looks nothing like ImageNet
appears to give a worse-conditioned starting point than random initialization** — the
model must first unlearn those features, and how cleanly that goes depends on the seed.

Three seeds is a weak basis for estimating a standard deviation, so this is an
observation rather than a finding. A five-fold difference is hard to attribute entirely
to chance.

### Why most of this negative result does not transfer

**The synthetic scene is smoothed Gaussian noise. It has no edges, no textures, no
object structure.** ImageNet's first layers are precisely edge, colour-blob and texture
detectors — and there is nothing here for them to detect. The experiment was close to
the worst possible case for ImageNet transfer, by construction.

Real satellite imagery does have edges and textures: coastlines, floe boundaries, field
margins, lead patterns. ImageNet pretraining usually does help there — less than Day 5's
CIFAR-10 result, but positively.

| finding | transfers? |
|---|---|
| dropping bands costs more than pretraining gains | **strongly** — the mechanism is informational, not architectural |
| the three adaptation strategies are indistinguishable | partly — on real imagery `inflate` typically edges ahead, though usually by little |
| ImageNet pretraining gave nothing | **weakly** — largely an artifact of textureless synthetic data |

The first row is the one to carry into the sea-ice project.

## Limitations

- **Three seeds per arm.** Enough to establish a 27-point gap and to rule out a 1-point
  one; not enough to estimate standard deviations precisely, which the variance
  observation above depends on.
- **Synthetic data with no texture.** Section 7 argues this makes the ImageNet result
  unrepresentative. The band-information result does not depend on it.
- **One encoder, one learning rate, 15 epochs, full fine-tuning throughout.** No
  frozen-backbone arm — which is where the `3/in_ch` rescaling should matter most, since
  BatchNorm cannot adapt to a magnitude shift it is not allowed to update. This notebook
  therefore under-tests the detail it spends the most text on.
- **`inflate` peaked at epoch 15** in the single-seed run, the last epoch available. The
  same epoch-cap censoring as Days 4, 8 and 9: these numbers are floors.
- **Four bands, not thirteen.** Averaging three RGB kernels is a weaker prior for
  Sentinel-2's band 11 than for a single NIR band, so inflation gets harder to justify as
  the channel count grows.
- **`hidden` caps the achievable score** by construction, as in Day 11. Absolute mIoU
  here is not comparable to anything outside this notebook.

## What follows

Day 5 showed ImageNet pretraining is worth more than architecture on natural images.
This notebook shows the obstacle to reusing it on satellite data is not the channel
count — that has three workable fixes — but that **ImageNet is the wrong source domain**,
and no amount of channel adaptation repairs a domain mismatch.

Remote-sensing pretrained backbones start from satellite data with the right channel
counts and no adaptation problem: **SSL4EO-S12** (self-supervised on Sentinel-1/2),
**Prithvi** (NASA/IBM, HLS), and BigEarthNet-pretrained models. That is the right default
for the sea-ice project. The techniques here are the fallback for a sensor no
domain-pretrained model covers — which, for passive microwave, is still most of them.